# MemGPT

Con MemGPT podemos hacer que un LLM, que está limitado por una ventana de contexto finita, parezca que tiene memoria infinita. Lo hace copiando el sistema de memoria virtual de los sistemas operativos. Igual que un sistema operativo pagina datos entre RAM y el disco, MemGPT pagina información entre el contexto del LLM y un almacenamiento externo.

## El LLM como la CPU de un sistema operativo

Está es una idea bastante extendida, de hecho fue dicha el 11 de noviembre de 2023 por Andrej Karpathy en un [tweet](https://x.com/karpathy/status/1723140519554105733)

Poco a poco hemos ido viendo como esto se hace cada vez más realidad, con la aparición de los harness, en los que el LLM es el nucleo ejecutor y a su alrededor les dotamos de tools, comandos, MCPs y skills

Últimamente además estamos viendo como empiezan a incorporar una gestión de la memoria.

Aquí vamos a ver esto último, replicando el paper [MemGPT: Towards LLMs as Operating Systems](https://arxiv.org/abs/2310.08560), en el que se crea

 * Una **jerarquía de memoria**: una pequeña rápida y una muy grande, pero lenta
 * Un **gestor de control de flujo**: que decide cuándo invocar al LLM y cuando gestionar la memoria
 * Un conjunto de **funciones**: que el propio LLM puede llamar para administrar su memoria

El LLM es **quien decide qué guardar**, qué buscar y qué descartar. No es un humano ni un sistema externo, es auto-dirigido.

### Modificación de la ventana de contexto

MemGPT modifica el contexto que se le manda al LLM

![MemGPT - ventana de contexto](https://images.maximofn.com/MemGPT-Context-window.webp)

#### System prompt

Al system prompt del LLM se le añade el system prompt de memGPT, en el que se le indica cómo se gestiona la memoria, y las funciones que le permitirán administrarla.

Esta sección es de solo lectura, el LLM ni memGPT la van a modificar

#### Working context

Bloque de tamaño fijo. Al principio está vacío o contiene solo la información de cómo es o cómo se tiene que comportar el agente

Aquí se irán guardando los datos importantes que tienen que estar en la ventana de contexto. Se guardan en lenguaje natural, no en datos estructurados tipo json

#### FIFO Queque

Es la zona de la ventana de contexto donde se irán añadiendo los mensajes de la conversación, las llamadas a tools, los datos obtenidos de las tools, etc.

Al inicio de esta FIFO, en la posición 0 se almacena un resumen del contexto. Cuando empieza la conversación está vacío, pero según avanza la conversación se va modificando con el resumen del FIFO

### Cómo se ve la ventana de contexto

En un agente normal, la ventana de contexto se vería así

```
Window: [
  system prompt;
  [(user) Hello;
   (agent) Hello, how can I help you?;
   (user) I have a question;
   (agent) About what?]
]
```

La misma conversación, con memGPT, se vería así

```
Window: [
  System Instructions (MemGPT prompt + memory description + function schemas);
  Working Context (empty or initial assistant);
  FIFO Queue [
    (recursive summary, empty);
    (user) Hello;
    (agent) Hello, what can I help you with?;
    (user) I have a question;
    (agent) About what?
  ]
]
```

### System Instructions

Corresponde al system prompt que el usuario escribe, pero al que se le añade el prompt de cómo se va a gestionar la memoria con MemGPT, cómo puede leer y escribir información en cada bloque y la información de las tools que memGPT le provee

### Qué es el Working Context

El **working context** es como la libreta o los posits del agente, aquí se guarda lo que el agente considera importante para tenerlo siempre en la ventana de contesto, pero además siempre al inicio de la ventana de contexto

#### Estructura del working context

En el paper original definieron el working context como un único bloque libre, en el que el LLM iba escribiendo lo que considerase

Sin embargo en la implementación de su código de [GitHub](https://github.com/letta-ai/letta/), lo definen como varios bloques separados, cada uno con una etiqueta y un texto.

Supongo que después de escribir el paper, y cuando probaron el código en varios clientes, vieron que era mejor de esta segunda manera, así que eso es lo que vamos a hacer

Un ejemplo de working context seria

| Label | What it stores | Example |
|---|---|---|
| `assistant` | Agent identity: name, personality, role, restrictions. | *"I'm Sam, a patient and curious assistant. I don’t reveal that I’m an AI.”* |
| `human` | Information about the user that the agent remembers. | *“The user’s name is Ana, she lives in Madrid, she likes surfing.”* |
| `custom` (any other) | Additional blocks defined by the developer. | `task_context`, `team_members`, `project_constraints`... |

Este cambio ofrece que

 * El agente pueda modificar un bloque sin modificar los otros
 * Cada bloque tiene un límite de tokens, lo que hace que la información del usuario no colapse la del asistente y viceversa
 * Es más estructurado, lo que permite reutilizar el working context del asistente o el custom en otros proyectos. Solo habría que añadir la información del usuario

#### Funciones de modificación del working context

Vamos a otorgar al LLM dos funciones para modificar el working context, una para añadir información y otra para reemplazarla. En ambas funciones hay que pasar el bloque (`assistant`, `human`, `custom`) y la información a añadir o a remplazar. Veamos dos ejemplos

 * `core_memory_append(label="human", content="Birthday is February 7")`
 * `core_memory_replace(label="human", old="Boyfriend named James", new="Ex-boyfriend named James")`

#### Propiedades clave del working context

| Propiedad | Explicación |
|---|---|
| **Tamaño fijo por bloque** | Cada bloque tiene su presupuesto de tokens, no puede crecer indefinidamente. |
| **Read/Write** | A diferencia de las System Instructions (read-only), el agente puede modificar cualquier bloque. |
| **Escritura solo vía funciones** | El LLM no escribe directamente generando texto; usa funciones tipo `core_memory_append(label, content)` o `core_memory_replace(label, old, new)`. |
| **Texto no estructurado dentro del bloque** | El contenido de cada bloque es texto libre en lenguaje natural, ni JSON ni esquemas rígidos. |
| **Persistente dentro de la ventana** | A diferencia de los mensajes de la FIFO Queue, **nunca se saca (expulsa) automáticamente por el harness**. Sigue ahí hasta que el LLM decida cambiarlo. |
| **Persistente entre sesiones** | Al reabrir una sesión nueva días después, todos los bloques se recargan tal y como quedaron. |

Como pone en la tabla, nunca se modifica automáticamente. Como veremos la FIFO se vaciará automáticamente en ciertas condiciones, sin que el LLM lo decida, lo hace memGPT, luego lo vemos.

Pero el working context solo se modifica cuando lo decide el LLM. El LLM decide qué quiere tener en este bloque como información importante y persistente. Como hemos dicho al inicio de explicar el working context, es su libreta de notas, o sus posits

#### Cuándo escribe el agente en el working context

El LLM escribe en el working context en dos situaciones

 1. Cuando ve un dato importante en la conversación y decide guardarlo.
 2. Cuando recibe la alerta *Memory Pressure*, más adelante lo explicamos, pero es una alerta que se le da al LLM indicando que la ventna de contexto se está llenando. Es como una manera de decirle al LLM, dentro de poco vamos a eliminar datos de la ventana de contexto, así que guarda lo que consideres importante, porque si no se va a perder

Cuando veamos cómo se gestiona cuando se llena el bloque FIFO, veremos que la información se guarda en archivos externos o en una base de datos. Dicha información guardada externamente, para recuperarla, el LLM la tiene que ir a buscar, por lo que puede fallar y que no la encuentre

Sin embargo, la información del working context, es información que el agente va a tener siempre en la ventana de contexto, y por tanto la va a tener siempre visible.

Además al colocarla al inicio de la ventana de contexto, hacemos que que el agente "le preste más atención". Ya que se ha visto que cuando la ventana de contexto empieza a crecer, los LLMs "hacen más caso" a la información que está al inicio de la ventana de contexto o al final, pero que la que está en medio, a veces no "se le hace mucho caso"

### Qué es la FIFO Queque

La FIFO Queque es la memoria donde se guarda toda la conversacisón del agente. Se guardan los mensajes del agente, del usuario, las llamadas a tools, los datos que devuelven las tools, alertas, etc. Todo se guarda en orden cronológico

Es una FIFO, porque veremos que cuando la ventana de contexto se llene y tengamos que eliminar información, eliminaremos los datos más antiguos, por eso es una FIFO (First In First Out)

#### Propiedades clave de la FIFO Queque

| Propiedad | Explicación |
|---|---|
| **Tamaño máximo dinámico** | No es de tamaño fijo absoluto, pero está acotado: lo que sobra de la ventana de contexto tras restar System Instructions y Working Context. |
| **Read/Write vía Queue Manager** | El LLM no escribe directamente en ella generando texto: es el Queue Manager quien añade entradas automáticamente. |
| **Orden cronológico** | Los elementos más antiguos están al principio, los más nuevos al final. Cuando se llena, se expulsan por el principio (FIFO). |
| **Slot 0 reservado** | El primer índice contiene siempre un *resumen recursivo* de los mensajes ya expulsados. Es el "qué pasó/qué ocurrió antes en la conversación" condensado por el LLM. |
| **Volátil pero recuperable** | Lo que se expulsa de la FIFO no se pierde: queda guardado en Recall Storage y puede traerse de vuelta (evocación) con `recall_storage.search(...)`. Más adelante lo explicaremos, pero son como los logs de la conversación |

#### Qué se guarda en la FIFO

En la FIFO se guarda toda la traza de la interacción:

 - **Mensajes del usuario**: lo que escribe el humano.
 - **Mensajes del agente al usuario**: respuesta del agente.
 - **Llamadas a funciones de MemGPT**: `working_context.append(...)`, `archival_storage.search(...)`, etc.
 - **Llamadas a cualquier otra tool del agente**: `web_search`, `calculator`, `bash`, etc.
 - **Salidas de todas esas funciones**: resultados de búsqueda, confirmaciones de éxito, errores.
 - **Mensajes de sistema**: alertas de Memory Pressure, eventos de login del usuario, notificaciones de uploads, etc.
 - **Eventos temporizados**: "son las 9:00 del 7 de febrero", para dar contexto al agente en el tiempo.

Un solo turno conversacional puede generar varias entradas en la FIFO (llamada + resultado + respuesta + más function chaining).

#### Umbrales de la FIFO

En el paper original se definen dos umbrales que disparan acciones:

| Umbral | Valor por defecto | Qué dispara |
|---|---|---|
| **Warning threshold** | ~70% del contexto | Inserta una *Memory Pressure Alert* en la FIFO para que el LLM consolide en Working Context lo que quiera preservar. |
| **Flush threshold** | ~100% del contexto | Dispara la expulsión: se elimina un bloque de mensajes (~50% del contexto por defecto) y se regenera el resumen recursivo. |

Son valores configurables, aunque en el paper hablan del 50%, 70% y 100%, nosotros podemos establecer los valores que queramos

Cuando llega un `warning threshold` (70%), el LLM sabe que dentro de poco se va a llegar al 100% y se va a eliminar mucha información de la ventana de contexto, por lo que tiene que ir guardando en el working context (la memoria dentro de la ventana de contexto) lo que considere importante para que no se pierda.

Cuando se llega al `flush threshold` (100%) de la ventana de contexto, se elimina un bloque de mensajes (~50% del contexto por defecto) y se regenera el resumen recursivo, el que está en el slot 0 de la FIFO, es decir, se genera un nuevo resumen, teniendo en cuenta lo que ya había antes en el resumen

#### El resumen recursivo (slot 0) y cómo se genera

Cuando se llega al `flush threshold` (100%) de la ventana de contexto:

 1. El Queue Manager identifica los mensajes a expulsar (los más antiguos, ~50% del contexto por defecto).
 2. Se coge el resumen que ya hay en el slot 0, el contenido del working context (la memoria) y los mensajes más antiguos
 3. Se hace una llamada separada al LLM, fuera del ciclo principal del agente, dedicada exclusivamente a regenerar el resumen y un nuevo working context
 4. El nuevo resumen ocupa el slot 0 de la FIFO.
 5. El nuevo working context se guarda en el working context del agente
 6. Los mensajes expulsados desaparecen de la FIFO pero permanecen en Recall Storage, de dónde se pueden recuperar (evocación).

Esta llamada de summarización es independiente porque **no cabría dentro del ciclo principal**: si el contexto del agente está al 100%, no se puede meter encima la instrucción `haz un resumen` más todo el material a resumir. Por eso el summarizer recibe su propio prompt mínimo, sin las System Instructions del agente principal.


Ejemplo e llamada al summarizer

```
System prompt:
  "You are a summariser. You receive:
    - The agent's current working context (what it already remembers effortlessly).
    - The previous recursive summary from the FIFO.
    - The oldest messages in the FIFO that are to be removed.
   Returns:
    - What to add to the Working Context (if there is new information
      worth persisting that is not already there).
    - The new recursive summary, incorporating the messages
      ejected from the previous summary."

Input:
  - Working Context: [current content]
  - Previous recursive summary: [FIFO slot 0]
  - Messages to be removed: [the oldest ~50%]

Output:
  - working_context_updates: [...] (may be empty)
  - new_recursive_summary: ‘...’
```

Pasarle el Working Context al summarizer es importante: así el resumidor **no propone añadir cosas que ya están guardadas**, y puede sugerir promociones de la FIFO al Working Context para datos relevantes a largo plazo.

#### Ejemplo de FIFO en mitad de una conversación

```
FIFO Queue [
  (system) Recursive summary: ‘The user's name is Ana, she lives in Madrid, she likes surfing...’;
  (user) ‘Do you remember where we went for lunch?’;
  (assistant) function_call: recall_storage.search(query=‘lunch pacifica’);
  (function_result) ‘[03/12/2024] “There’s a Taco Bell next to the beach”’;
  (assistant) function_call: send_message(‘Yes! We went to the Taco Bell in Pacifica.’);
  (function_result) {‘status’: ‘ok’};
  (user) ‘That’s right! By the way, I’m going with James tomorrow’;
  (system) Memory Pressure Warning: context at 75%;
  (assistant) function_call: working_context.append(‘You have a friend/partner called James’);
  (function_result) {‘status’: ‘ok’};
  (assistant) function_call: send_message(‘Great! Fancy going to Taco Bell again?’);
  (function_result) {‘status’: ‘ok’};
]
```

Si el Working Context es **lo que el agente recuerda sin esfuerzo** y Recall/Archival/MemFS es **lo que tiene que ir a buscar**, la FIFO Queue es **el flujo en vivo de lo que está pasando ahora mismo**.

### Eventos automáticos

MemGPT puede dispararse automáticamente por varios tipos de eventos. Como hemos dicho memGPT es un envoltorio que gestiona al LLM, por lo que vamos a poder hacer que se ejecuten acciones en un momento determinado, tras varias iteraciones, durante varias iteraciones, etc.

#### Eventos temporizados (`wall-clock`)

Podemos hacer que el agente realice una acción a una hora determinada o en intervalos regulares.

Ejemplos:

 * Todos los días a una hora determinada: "Da los buenos días al usuario todos los días a las 09:00"
 * En un intervalo de tiempo: "Manda el correo dentro de 30 minutos"
 * Recursivamente: "Cada 10 minutos comprueba el estado de mi aplicación"

#### Eventos por iteraciones (`sleep-time agents`)

En lugar de medir tiempo real, se cuentan **pasos del agente principal**: cada N inferencias o cada N turnos del usuario, se dispara un agente auxiliar (un *sleep-time agent*).

```
Principal agent → step 1 → step 2 → step 3 → ... → step N
                                                     │
                                                     ▼
                                          Sleep-time activate the agent
                                          (review memory, consolidate,
                                           sugiere acciones, etc.)
```

Casos de uso típicos:

 * Consolidación periódica de la memoria: "Cada 10 turnos, despierta a un agente que revisa la conversación y decide si hay que almacenar nuevos datos en el working context"
 * Revisión de la calidad de las respuestas: "Cada 20 mensajes, despierta a un agente que revisa si la assistant del agente principal sigue siendo coherente con el comportamiento observado"
 * Mantenimiento de calidad: "Tras 20 mensajes sin actividad de búsqueda, despierta a un agente que sugiera buscar en archival si parece que falta contexto"
 * Ejecución sin dependencia de un proceso vivo: "Cada 50 turnos, despierta un agente que revise si la assistant del agente principal sigue siendo coherente con el comportamiento observado"

### Encadenar acciones con `request_heartbeat`

Esto es algo que ya no es necesario con los últimos modelos de LLMs, pero lo explicamos por si usas un modelo con pocas capacidades

Es un **argumento booleano especial** que el LLM puede incluir en cualquiera de sus llamadas a funciones. No afecta a la lógica de la función, es una instrucción **para MemGPT**, no para la función:

Imagina que pides al agente que busque dónde está definida una función en tu código. El agente puede buscar en un script, si no encuentra busca en otro y así hasta que encuentre la función

Este sería el comportamiento ideal, pero con los LLMs más antiguos o menos capaces, llegaba un momento en el que al LLM le daba pereza y paraba

Con este booleano, aunque el LLM se pare, memGPT volverá a llamar a la tool (en el caso anterior la de búsqueda) y seguirá ejecutando el flujo.

#### Riesgo de bucles infinitos

Sin embargo este método tiene el riesgo de encadenar bucles infinitos, así que para evitarlo, se suelen añadir las siguientes protecciones

 * **Límite máximo de heartbeats encadenados por turno** (p. ej. 10).
 * **Timeout** total por turno.
 * **Detección de loops** (mismas llamadas repetidas sin progreso).

#### LLMs modernos

Como hemos dicho, con LLMs modernos esto ya no es necesario, por lo que lo que hacemos es que si el LLM devuelve que hay que usar una tool, se llama a la tool, si no devuelve que hay que usar una tool, se considera que es la respuesta al usuario y se termina el bucle

### `Recall Storage`: memoria de largo plazo

El **Recall Storage** es el **historial completo de la conversación**. Vive **fuera de la ventana de contexto** y guarda **todos los mensajes** que han pasado por la FIFO Queue, también los que ya fueron expulsados. Es el "log" de la sesión: lo que se ha dicho, palabra por palabra.

#### Qué guarda el Recall Storage

Cada mensaje (`HumanMessage`, `AIMessage`, `ToolMessage`, etc.) que entra en la FIFO se copia automáticamente a Recall en el mismo momento. No se hace cuando la ventana de contexto llega al 100% y se hace el flush.

Así, cuando se hace el flush, los mensajes que se expulsan de la FIFO no se pierden porque ya están guardados en el `Recall Storage`.

#### Quién escribe en el Recall Storage

En el Recall Storage escribe el **Queue Manager**, automáticamente. **El LLM no llama a ninguna función para escribir en Recall**, pasa solo. Es como un log pasivo del sistema, no una decisión del agente.

#### Cómo usa el agente el Recall Storage

El LLM tiene una única función para usar el Recall Storage y es la de recuperar mensajes.

```
recall_memory_search(query="...", page=N)   # search in the history
```

Realiza la búsqueda mediante RAG. El contenido se ha convertido en embeddings, la búsqueda se convierte también en un embedding y se hace una búsqueda por similitud.

#### Propiedades clave del Recall Storage

| Propiedad | Explicación |
|---|---|
| **Tamaño ilimitado** | Toda la historia conversacional, sin límite. |
| **Escritura automática** | El LLM no decide qué entra: entra todo lo que pase por la FIFO. |
| **Solo lectura para el LLM** | El agente puede *buscar* pero no insertar ni borrar. |
| **Búsqueda semántica + paginación** | Encoder + similaridad + páginas bajo demanda. |
| **Persistente entre sesiones** | Al reabrir el agente días después, todo el historial sigue siendo buscable. |

### `Archival Storage`: memoria de largo plazo

El **Archival Storage** es la memoria de largo plazo del agente: vive **fuera de la ventana de contexto**, es **ilimitada en tamaño** y el LLM accede a ella **solo bajo demanda** mediante funciones de búsqueda. Es el análogo al "disco" del SO: lenta de acceder, pero cabe todo.

#### Qué guarda el Archival Storage

Texto arbitrario que el propio agente decide persistir: resúmenes que él mismo redacta, hechos sueltos que quiere recordar a largo plazo, documentos largos que el usuario le aporta, conclusiones que extrae de una conversación.

**No** se guarda automáticamente cada mensaje (eso lo hace Recall Storage, ver más arriba). Archival es escritura **explícita**: solo entra lo que el LLM llama `archival_memory_insert(...)`.

#### Diferencia entre Recall y Archival storage

La diferencia entre Recall y Archival storage es que el Recall storage guarda todo el historial de la conversación, mientras que el Archival storage guarda solo lo que el propio agente decide persistir.

#### Cómo se usa el Archival Storage

El agente tiene dos funciones para usar el Archival Storage:

```
archival_memory_insert(content="...")             # write
archival_memory_search(query="...", page=N)       # read (vector search, paginated)
```

La búsqueda es **semántica**: el `storage` indexa cada entrada con un encoder, es decir, mediante embeddings y `search` devuelve los top-K más similares al `query`. Si la página 1 no convence, el LLM pagina con `page=2` y reformula si hace falta

Es **retrieval iterativo y auto-dirigido**, no un único disparo. Es decir, si el LLM no encuentra lo que busca, puede volver a hacer la búsqueda con una nueva query, o con la misma query pero con un nuevo page, etc.

#### Propiedades clave del Archival Storage

| Propiedad | Explicación |
|---|---|
| **Tamaño ilimitado** | A diferencia del Working Context, no hay límite de tokens, crece todo lo que haga falta. |
| **Read/Write vía funciones** | El LLM controla qué entra (`insert`) y qué sale (`search`). Nada se inyecta automáticamente. |
| **Búsqueda semántica** | Embeddings + similaridad por coseno. No es `grep`. Si se implementa con `substring match` deja de ser MemGPT. |
| **Paginación** | Los resultados se sirven en páginas; el agente decide cuándo seguir pidiendo. Esto es lo que evita el "lost in the middle". |
| **Persistente entre sesiones** | Se serializa a una BD vectorial (pgvector, Chroma, FAISS…). Sobrevive reinicios. |

#### Diferencia entre la búsqueda del archival y RAG clásico

Archival usa **RAG por debajo** (embeddings + similaridad), pero se diferencia en que el LLM **decide** cuándo buscar, qué buscar y cuantas veces buscar.

En RAG clásico, el pipeline recupera siempre, mete los top-K en el prompt y el corpus es estático.

**RAG clásico** ("chatear con tus PDFs"):

1. Trocear documentos en chunks.
2. Embeber cada chunk → vector.
3. Indexar en una BD vectorial (Chroma, FAISS, pgvector…).
4. En cada query del usuario: embeber la pregunta → buscar top-K chunks → **inyectarlos automáticamente en el prompt** → el LLM genera respuesta.

En RAG clásico, **el pipeline decide buscar**: el usuario pregunta, el sistema recupera, el LLM solo lee y responde. La recuperación es **opaca y de un solo paso** - lo que entra al prompt es lo que entra, no hay vuelta atrás.

**Archival Storage** hace exactamente lo mismo (vector search sobre embeddings), pero la búsqueda está expuesta como **tool** (`archival_memory_search(query, page)`) que el **LLM decide** invocar. Eso cambia tres cosas:

 1. **El agente decide si necesita buscar.**Si la pregunta es trivial, no llama a la tool y se ahorra la búsqueda. RAG clásico siempre recupera, aunque sea una pregunta sencilla y no sea necesario.
 2. **El agente puede paginar.** Si la primera página de resultados no convence, llama a la tool de nuevo con `page=2`, o reformula la query. Con K alto (muchos docs distractores), MemGPT mantiene buenos resultados porque pagina; el RAG clásico cae porque mete los top-K en el prompt y el LLM se pierde con tanta información.
 3. **El agente puede escribir en Archival Sotrage.** `archival_memory_insert` deja al LLM persistir conocimiento que generó él mismo (resúmenes, conclusiones, lo que el usuario le contó hace 3 sesiones). En RAG clásico la documentación es estática.

En resumen: **Archival = RAG (motor de retrieval) + control del LLM (cuándo, cómo, cuántas veces) + escritura por el agente**. La diferencia no está en cómo se almacena, está en quién decide cuándo recuperar y qué hacer si lo recuperado no basta.

### MemFS - memoria en file system versionado

Aunque en el paper original no utilizaban el sistema de ficheros como un recurso de memoria a largo plazo, se ha visto en los últimos sistemas de agentes, que los comandos `bash` funcionan muy bien, y por tanto poder guardar la información en archivos en local también

Se guarda la memoria a largo plazo en un sistema de ficheros versionado con git. De esta manera puede consultar el historial completo de cambios

#### Cómo usa el agente memFS

El agente trata memFS como un filesystem con herramientas como

```
memfs.create(path="/projects/web/notes.md", content="...")
memfs.read(path="/projects/web/notes.md")
memfs.write(path="/projects/web/notes.md", content="...")
memfs.list(path="/projects/web/")
memfs.move(src="...", dst="...")
memfs.history(path="...")
memfs.grep(path="...", query="...")
```

#### Por qué da mejores resultados este tipo de memoria a largo plazo

 * **Proyectos largos**: organizar el conocimiento del agente en una jerarquía rica (`/users/ana/`, `/projects/portfolio/`, `/learning/spanish/`).
 * **Trazabilidad**: poder ver **cómo evolucionó** una pieza de información ("¿qué decía el bloque del usuario hace 3 semanas?").
 * **Rollback**: si el agente sobrescribió algo importante por error, se puede recuperar la versión anterior.
 * **Compartir contexto entre agentes**: en sistemas multi-agente, varios agentes pueden leer/escribir sobre el mismo MemFS, usándolo como pizarra compartida.

### Working context vs archival storage vs memFS

En esta comparación no metemos Recall Storage, porque como hemos dicho es la información de la FIFO que se almacena sin que el LLM lo decida, son como los logs

Los tres son **memoria persistente del agente**, pero optimizan ejes distintos. **No son alternativas: coexisten**, y el agente decide caso a caso dónde guarda cada cosa nueva.

| Dimensión | Working Context (Core Memory) | Archival Storage | MemFS |
|---|---|---|---|
| **Dónde vive** | Dentro de la ventana de contexto | Fuera (BD vectorial) | Fuera (filesystem versionado) |
| **Tamaño** | Pequeño (budget de tokens) | Ilimitado | Ilimitado |
| **En contexto siempre** | ✅ Sí, visible en cada inferencia | ❌ No (se busca) | ❌ No (se navega) |
| **Cómo se accede** | Lectura directa (está en el prompt) | `archival_memory_search` (vector search) | `memfs.read` / `memfs.list` / `memfs.grep` (por path) |
| **Cómo se escribe** | `core_memory_append/replace` por bloque | `archival_memory_insert` | `memfs.create/write/move` |
| **Estructura** | Bloques etiquetados planos (`assistant`, `human`, …) | Entradas sueltas indexadas por embedding | Árbol jerárquico de ficheros y carpetas |
| **Búsqueda** | No hace falta (se ve directo) | Semántica (similaridad) | Estructural (path) + por contenido |
| **Versionado** | ❌ No | ❌ No | ✅ Sí, tipo git |
| **Coste por uso** | Tokens del contexto en cada turno | Una o varias tool call | Una o varias tool call |

#### Cómo tiene que decidir el LLM qué va en cada memoria

| Si el dato es… | Va a… |
|---|---|
| Crítico, debe verse en **cada** turno (identidad, preferencias clave, estado actual del usuario) | **Working Context** |
| Voluminoso, no sabes a priori cuándo lo necesitarás, búsqueda por significado ("¿qué hablamos de X hace meses?") | **Archival Storage** |
| Estructurado, jerárquico, evoluciona en el tiempo y quieres ver su historial (notas de proyecto, especificaciones, bitácoras) | **MemFS** |

Veamos un ejemplo

 * **Working Context**: `"El usuario es Máximo, prefiere español, trabaja con Astro + LangGraph"` → siempre visible.
 * **Archival Storage**: cada conversación importante resumida y persistida (`archival_memory_insert("El usuario decidió usar pgvector porque ...")`); luego `archival_memory_search("decisión sobre vector DB")` la recupera meses después.
 * **MemFS**:
  - `/projects/portafolio/decisiones.md` → bitácora versionada de decisiones de arquitectura
  - `/users/maximo/objetivos-q2.md` → objetivos del trimestre
  - `memfs.history("/projects/portafolio/decisiones.md")` permite ver cómo evolucionaron las decisiones.

## Ciclo de funcinamiento

### Ciclo de funcionamiento cuando llega un evento

Cuando llega un evento, como un mensaje de usuario, una alerta del sistema, un evento programado, etc, lo que ocurre es lo siguiente

 1. **Queue Manager** añade el mensaje a la FIFO Queue y lo guarda en Recall Storage (evocación).
 2. Concatena los tres bloques del main context (system prompt + working context + queque FIFO) y dispara la inferencia del LLM.
 3. Si la salida no contiene una tool, se manda al usuario, si contiene una tool se ejecuta la tool.
 4. Si se tiene que ejecutar una tool, la tool se ejecuta (puede ser: escribir en working context, buscar en archival, etc.) y su resultado se devuelve al main context.
 5. Se vuelve a repetir el bucle con la salida de la main tool

### Ciclo de funcionamiento cuando ocurre un desbordamiento de la ventana de contexto

Cuando se llega al 70% o al 100% de la ventana de contexto

 - Cuando los tokens superan el **warning threshold** (~70%), se inserta una *Memory Pressure Alert* en la FIFO. El LLM, viendo la alerta, decide qué guardar de la FIFO en working context, en el archival o en MemFS antes de perderlo.
 - Cuando se supera el **flush threshold** (100%), el Queue Manager evicta (expulsa) mensajes antiguos, genera un nuevo resumen recursivo, un nuevo working context y se libera espacio. Lo expulsado **no se pierde**: sigue en Recall Storage y se puede recuperar con búsquedas paginadas.

## Resumen

MemGPT convierte al LLM en un agente que se auto-administra la memoria mediante function calls, con una jerarquía de dos niveles (contexto = RAM, archival/recall/MemFS (documental/evocación) = disco) y un control de flujo dirigido por eventos, logrando que un modelo de 8k tokens se comporte como si tuviera contexto ilimitado